# Accesso ai File dal Codice Python

Uno dei compiti più comuni nel lavoro di uno sviluppatore è l'elaborazione di dati memorizzati all'interno di file. Questi ultimi sono solitamente conservati all'interno di dispositivi fisici di archiviazione, come dischi rigidi (HDD), unità a stato solido (SSD), supporti ottici o dischi di rete.

È facile immaginare un programma che ordina venti numeri, così come è altrettanto semplice pensare a un utente che inserisce questi venti numeri direttamente a mano usando la tastiera. Tuttavia, diventa molto più difficile immaginare lo stesso compito quando i numeri da ordinare diventano 20.000: nessun utente sarebbe in grado di digitarli tutti senza commettere almeno un errore.

In uno scenario del genere, la soluzione ideale è memorizzare i numeri in un file sul disco, che verrà poi letto dal programma. Il software ordinerà i dati e, invece di visualizzarli sullo schermo, creerà un nuovo file in cui salverà la sequenza numerica ordinata.

Se vogliamo implementare un database, anche semplice, l'unico modo per conservare le informazioni tra una sessione di esecuzione e l'altra del programma consiste nel salvarle in un file (o in più file, se la struttura della base dati è complessa). In linea di principio, qualsiasi problema di programmazione non banale si affida all'uso dei file: che si tratti di elaborare immagini, moltiplicare matrici o calcolare stipendi e tasse leggendo record storici.

Python implementa l'accesso e l'elaborazione dei file attraverso un insieme coerente e strutturato di oggetti.


## Nomi dei file

I diversi sistemi operativi gestiscono i file in modi differenti. Ad esempio, Windows adotta una convenzione di denominazione diversa rispetto a quella utilizzata nei sistemi Unix/Linux.

Se consideriamo il concetto di **nome canonico del file** (ovvero il percorso assoluto che definisce univocamente la posizione del file indipendentemente dalla cartella corrente), notiamo subito una discrepanza visiva tra i sistemi:

* **Windows:** Utilizza la lettera dell'unità seguita dai backslash, ad esempio `C:\cartella\file.txt`
* **Unix/Linux:** Utilizza una struttura ad albero unica che parte dalla radice, definita da slash puri, ad esempio `/utente/cartella/file.txt`

Inoltre, i nomi dei file nei sistemi Unix/Linux sono **case-sensitive** (distinguono tra lettere maiuscole e minuscole). I sistemi Windows, al contrario, conservano visivamente le maiuscole usate al momento della creazione del file, ma non effettuano distinzioni quando vi si accede. Di conseguenza, le stringhe `QuestoEIlNomeDelFile` e `questoeilnomedelfile` descrivono due file completamente diversi su Unix/Linux, mentre indicano lo stesso identico file su Windows.

La differenza più evidente risiede nel separatore delle cartelle: `\` in Windows e `/` in Unix/Linux. Questa diversità è cruciale quando si scrivono programmi in Python a causa del ruolo speciale del carattere `\` all'interno delle stringhe (dove funge da carattere di escape, come nel caso di `\n` per la nuova riga).

Se su Windows provi a definire un percorso in questo modo:

```python
nome = "\dir\file"

```

Andrai incontro a comportamenti imprevisti o errori di sintassi, poiché Python interpreterà i caratteri iniziali dopo il backslash come sequenze speciali. Per ovviare al problema su Windows, è necessario raddoppiare i backslash:

```python
nome = "\\dir\\file"

```

In alternativa, Python è abbastanza intelligente da convertire automaticamente gli slash in backslash sotto il cofano quando rileva che il sistema operativo sottostante lo richiede. Di conseguenza, le seguenti assegnazioni funzioneranno correttamente anche su Windows:

```python
nome = "/dir/file"
nome_unita = "c:/dir/file"

```

I programmi non comunicano mai direttamente con i file fisici, ma lo fanno attraverso entità astratte chiamate **handle** o **stream** (flussi di dati). Sfruttando i metodi messi a disposizione da Python, il programmatore esegue operazioni sullo stream, le quali vengono tradotte in modifiche sul file reale tramite il kernel del sistema operativo.

Per collegare uno stream a un file fisico, è necessario eseguire un'operazione esplicita chiamata **apertura del file** (`open`). Quando il legame non serve più, lo stream viene scollegato tramite la **chiusura del file** (`close`). Qualsiasi manipolazione dei dati avviene sempre all'interno di questa finestra temporale.

L'apertura di un file può fallire per diversi motivi: il file potrebbe non esistere, il programma potrebbe non disporre dei permessi di lettura/scrittura necessari, oppure il sistema operativo potrebbe aver raggiunto il limite massimo di stream contemporanei consentiti (ad esempio, 200 file aperti nello stesso momento). Un programma ben scritto deve sempre intercettare questi fallimenti e reagire di conseguenza per evitare crash improvvisi.

## Stream di file

L'apertura di uno stream richiede la dichiarazione della modalità di elaborazione dei dati, chiamata **modalità di apertura** (*open mode*). Se l'apertura va a buon fine, il programma potrà eseguire solo le operazioni permesse da quella specifica modalità.

Le operazioni fondamentali su uno stream sono due:

* **Lettura dallo stream:** Porzioni di dati vengono recuperate dal file e caricate nella memoria gestita dal programma (ad esempio, all'interno di una variabile).
* **Scrittura sullo stream:** Porzioni di dati vengono trasferite dalla memoria del programma all'interno del file fisico.

Le tre modalità di apertura principali sono:

* **Modalità di lettura (read mode):** Consente solo operazioni di lettura. Qualsiasi tentativo di scrittura solleverà l'eccezione `UnsupportedOperation` (che eredita da `OSError` e `ValueError` del modulo `io`).
* **Modalità di scrittura (write mode):** Consente solo operazioni di scrittura. Tentare di leggere dallo stream solleverà la medesima eccezione.
* **Modalità di aggiornamento (update mode):** Consente di eseguire sia operazioni di lettura che di scrittura sullo stesso stream.

Lo stream si comporta in modo analogo a un vecchio registratore a nastro. Quando leggi o scrivi dati, una **testina virtuale** si sposta lungo il file in base al numero di byte trasferiti. Nei manuali di programmazione questo meccanismo viene definito **posizione corrente del file** (*current file position*).

## 4.2.4 Handle dei file

In Python, ogni file è gestito tramite un oggetto appartenente a una classe specifica, determinata dalle intenzioni del programmatore e dal tipo di contenuto del file stesso. Questi oggetti vengono creati all'atto dell'apertura del file e distrutti al momento della chiusura.

Non si utilizzano mai i costruttori diretti per istanziare queste classi; l'unico modo per ottenerli è invocare la funzione nativa `open()`. Python analizzerà gli argomenti forniti e istanzierà l'oggetto corretto (per i nostri scopi, ci concentreremo principalmente sugli stream di tipo `BufferedIOBase` e `TextIOBase`). Per distruggere l'oggetto e recidere il legame con il file, si invoca il metodo `.close()`.

In base al tipo di contenuto, gli stream si dividono in due macro-categorie:

* **Stream di testo:** Sono strutturati in righe e contengono caratteri tipografici leggibili (lettere, cifre, punteggiatura). Vengono letti o scritti carattere per carattere o riga per riga.
* **Stream binari:** Non contengono testo strutturato, ma una sequenza grezza di byte di qualsiasi valore (es. immagini, file audio, programmi eseguibili, database). Mancando la suddivisione in righe, la lettura e la scrittura avvengono per blocchi di dimensioni arbitrarie o byte per byte.

Sorge qui un problema di portabilità legato ai caratteri di fine riga (newline):

* I sistemi **Unix/Linux** marcano la fine di una riga con un singolo carattere chiamato **LF** (Line Feed, codice ASCII 10), rappresentato in Python come `\n`.
* I sistemi **Windows** utilizzano una coppia di caratteri: **CR** (Carriage Return, codice ASCII 13) seguito da **LF**, codificati come `\r\n`.

Se un programma scritto per Windows cercasse la sequenza `\r\n` per identificare le righe, diventerebbe inutilizzabile su Linux, e viceversa. Per garantire la **portabilità** dei programmi, Python risolve questo problema alla radice in modo completamente automatico attraverso le classi degli stream di testo:

1. Quando un file viene aperto in **modalità testo**, il sistema attiva la traduzione automatica dei caratteri di fine riga.
2. In fase di **lettura** su Windows, ogni coppia `\r\n` viene convertita in un singolo carattere `\n`. In fase di **scrittura**, ogni carattere `\n` inserito dal programma viene automaticamente convertito nella coppia `\r\n` sul file fisico.
3. Nei sistemi Unix/Linux non avviene alcuna conversione poiché il formato nativo coincide già con lo standard di Python. Questo meccanismo trasparente permette di scrivere il codice una sola volta e vederlo funzionare ovunque.
4. Se il file viene aperto in **modalità binaria**, i dati vengono letti e scritti esattamente così come sono (*as-is*), senza alcuna conversione.

## Apertura degli stream

La sintassi generale per aprire un file è la seguente:

```python
stream = open(file, mode='r', encoding=None)

```

* **`file`**: Stringa che indica il percorso e il nome del file da associare.
* **`mode`**: Stringa contenente uno o più caratteri che specificano la modalità di accesso. Se omesso, il valore predefinito è `'r'` (lettura in modalità testo).
* **`encoding`**: Specifica la codifica dei caratteri (ad esempio `'utf-8'`) quando si lavora in modalità testo. La codifica predefinita dipende dalla piattaforma in uso.

Se l'operazione ha successo, la funzione restituisce l'oggetto stream; in caso contrario, solleva un'eccezione (come `FileNotFoundError`).

### Riepilogo delle modalità di apertura principali:

* **`'r'` (Read):** Il file deve esistere ed essere leggibile, altrimenti viene sollevata un'eccezione. La testina virtuale si posiziona all'inizio del file.
* **`'w'` (Write):** Se il file non esiste, viene creato. Se esiste, il suo contenuto precedente viene completamente cancellato (lunghezza troncata a zero).
* **`'a'` (Append):** Se il file non esiste, viene creato. Se esiste, la testina si posiziona alla fine del file, preservando intatto il contenuto preesistente e permettendo l'aggiunta di nuovi dati.
* **`'r+'` (Read + Update):** Il file deve esistere. Permette sia la lettura che la scrittura sullo stream.
* **`'w+'` (Write + Update):** Se il file non esiste, viene creato. Permette sia la lettura che la scrittura, ma cancella il contenuto precedente del file all'atto dell'apertura.


## Selezione della modalità testo o binaria

Per esplicitare il tipo di stream, si aggiunge un carattere alla fine della stringa della modalità:

* **`'b'`**: Attiva la modalità **binaria** (es. `'rb'`, `'wb'`).
* **`'t'`**: Attiva la modalità **testo** (es. `'rt'`, `'wt'`). È il comportamento predefinito.

> **Extra:** Esiste anche la modalità **`'x'`** (Exclusive creation). Consente di aprire un file per la scrittura solo ed esclusivamente se il file non esiste già sul disco. Se il file è presente, `open()` solleverà immediatamente un'eccezione, impedendo sovrascritture accidentali.

## Aprire uno stream per la prima volta

Ecco un esempio pratico e sicuro per aprire in lettura un file di testo situato su Windows all'interno di un blocco di gestione degli errori `try-except`:

```python
try:
    stream = open("C:/Users/User/Desktop/file.txt", "rt")
    # Qui avviene l'elaborazione dei dati
    stream.close()
except Exception as exc:
    print("Impossibile aprire il file:", exc)

```

Usiamo gli slash puri per evitare conflitti con i caratteri di escape e intercettiamo qualsiasi anomalia a runtime stampando l'errore per capire esattamente cosa sia andato storto.

## Stream pre-aperti

Esistono tre eccezioni alla regola secondo cui ogni stream deve essere esplicitamente aperto prima dell'uso. Quando un programma Python si avvia, il sistema operativo apre automaticamente tre stream standard, accessibili importando il modulo `sys`:

* **`sys.stdin` (Standard Input):** Associato di norma alla tastiera, è aperto in sola lettura e rappresenta la sorgente primaria di dati per il programma. La funzione integrata `input()` legge da questo stream.
* **`sys.stdout` (Standard Output):** Associato di norma allo schermo, è aperto in sola scrittura e rappresenta il canale principale per mostrare i risultati. La funzione `print()` invia i dati a questo stream.
* **`sys.stderr` (Standard Error):** Associato anch'esso allo schermo e aperto in scrittura, è il canale dedicato in cui convogliare i messaggi di errore e di diagnostica del programma.

Separare l'output dei risultati (`stdout`) dall'output degli errori (`stderr`) permette di reindirizzare queste informazioni verso destinazioni differenti (ad esempio, salvare gli errori in un file di log separato lasciando i risultati puliti sullo schermo).

## Chiusura degli stream

L'ultima operazione da eseguire su uno stream aperto (esclusi quelli standard) è la chiusura tramite il metodo `stream.close()`. Questa funzione non richiede argomenti e non restituisce nulla, ma può sollevare un'eccezione `IOError` in caso di fallimento.

Molti programmatori tendono a non controllare l'esito di un `close()`, dando per scontato che vada sempre a buon fine. Questo è parzialmente rischioso: quando si scrive su un file, Python memorizza temporaneamente i dati in un'area di transito chiamata **buffer** (caching). Il metodo `.close()` forza il sistema a svuotare i buffer scrivendo i dati residui sul disco fisco (*flush*); se questa operazione fallisce (es. disco pieno o disconnesso all'improvviso), anche la chiusura del file fallirà.

## Diagnosticare i problemi dello stream

L'oggetto eccezione di tipo `IOError` offre una proprietà chiamata `errno` (error number) che permette di identificare numericamente la causa esatta del fallimento. Questo valore può essere confrontato con le costanti simboliche contenute all'interno del modulo integrato `errno`.

Ecco le costanti più comuni utilizzate per la diagnostica:

* **`errno.EACCES` (Permission denied):** Si verifica quando si tenta di scrivere su un file con attributi di sola lettura o protetto dai permessi di sistema.
* **`errno.EBADF` (Bad file number):** Si verifica quando si tenta di operare su uno stream non valido o già chiuso.
* **`errno.EEXIST` (File exists):** Si verifica ad esempio quando si tenta di rinominare un file usando il nome di un file già presente.
* **`errno.EFBIG` (File too large):** Si verifica quando il file supera le dimensioni massime consentite dal file system.
* **`errno.EISDIR` (Is a directory):** Si verifica quando si tenta di trattare una cartella come se fosse un file normale.
* **`errno.EMFILE` (Too many open files):** Si verifica quando il programma supera il limite di file aperti simultaneamente consentiti dal sistema operativo.
* **`errno.ENOENT` (No such file or directory):** Si verifica quando si fa riferimento a un file o a un percorso inesistente.
* **`errno.ENOSPC` (No space left on device):** Si verifica quando lo spazio di archiviazione sul dispositivo è esaurito.

Un programmatore attento può strutturare un controllo dettagliato combinando questi elementi:

In [1]:
import errno

try:
    s = open("c:/users/user/Desktop/file.txt", "rt")
    # Actual processing goes here.
    s.close()
except Exception as exc:
    if exc.errno == errno.ENOENT:
        print("The file doesn't exist.")
    elif exc.errno == errno.EMFILE:
        print("You've opened too many files.")
    else:
        print("The error number is:", exc.errno)
        

The file doesn't exist.


Fortunatamente, esiste una funzione in grado di semplificare drasticamente il codice dedicato alla gestione degli errori.

Il suo nome è `strerror()`, fa parte del modulo `os` e richiede un solo argomento: il numero dell'errore (`errno`). Il suo compito è semplicissimo: tu le passi il codice numerico dell'errore e lei ti restituisce una stringa di testo che ne descrive il significato in linguaggio umano.

> **Nota:** Se passi alla funzione un codice di errore inesistente (ovvero un numero che non è associato a nessun errore reale del sistema), verrà sollevata un'eccezione di tipo `ValueError`.

Ora possiamo semplificare il nostro codice precedente nel seguente modo:


In [2]:
import os
import errno

try:
    stream = open("dati.txt", "rt")
    # Qui avvengono le operazioni sullo stream
    stream.close()
except IOError as exc:
    # os.strerror converte automaticamente il codice in una descrizione chiara
    print("Si è verificato un errore:", os.strerror(exc.errno))

Si è verificato un errore: No such file or directory


Grazie a questo approccio, non è più necessario inserire lunghe catene di istruzioni `if-elif` per controllare manualmente ogni singola costante del modulo `errno`. Ci penserà direttamente la funzione `os.strerror()` a produrre il messaggio diagnostico appropriato (come *"No such file or directory"* o *"Permission denied"*), rendendo lo script molto più compatto, pulito e facile da mantenere.

In [3]:
from os import strerror

try:
    s = open("c:/users/user/Desktop/file.txt", "rt")
    # Actual processing goes here.
    s.close()
except Exception as exc:
    print("The file could not be opened:", strerror(exc.errno))

The file could not be opened: No such file or directory
